In [2]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tensorflow as tf
import numpy as np
from transformers import GPT2Tokenizer

In [3]:
tokenizer=GPT2Tokenizer.from_pretrained('gpt2')

In [4]:
# simple Dataset
inputs = tf.data.Dataset.from_tensor_slices(  [[0,1,2],[1,2,3]])
targets = tf.data.Dataset.from_tensor_slices( [[1,2,3],[2,3,4]])
dataset = tf.data.Dataset.zip(inputs, targets)
list(dataset.as_numpy_iterator())

[(array([0, 1, 2], dtype=int32), array([1, 2, 3], dtype=int32)),
 (array([1, 2, 3], dtype=int32), array([2, 3, 4], dtype=int32))]

In [5]:
# simple Dataset
tokens = [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19]
inputs  = tf.data.Dataset.from_tensor_slices(  [tokens[:3], tokens[1:4]])
targets = tf.data.Dataset.from_tensor_slices(  [tokens[1:4], tokens[2:5]])
dataset = tf.data.Dataset.zip(inputs, targets)
list(dataset.as_numpy_iterator())

[(array([0, 1, 2], dtype=int32), array([1, 2, 3], dtype=int32)),
 (array([1, 2, 3], dtype=int32), array([2, 3, 4], dtype=int32))]

In [130]:
# I can't get this windows thing to work right..
tokens = [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19]
inputs = tf.data.Dataset.from_tensor_slices(tokens).window(4, shift=1, drop_remainder=True)
targets = tf.data.Dataset.from_tensor_slices(tokens[1:]).window(4, shift=1, drop_remainder=True)
dataset = tf.data.Dataset.zip(inputs, targets)
for window in dataset:
    print(list(window[0].as_numpy_iterator()), list(window[1].as_numpy_iterator()))

[0, 1, 2, 3] [1, 2, 3, 4]
[1, 2, 3, 4] [2, 3, 4, 5]
[2, 3, 4, 5] [3, 4, 5, 6]
[3, 4, 5, 6] [4, 5, 6, 7]
[4, 5, 6, 7] [5, 6, 7, 8]
[5, 6, 7, 8] [6, 7, 8, 9]
[6, 7, 8, 9] [7, 8, 9, 10]
[7, 8, 9, 10] [8, 9, 10, 11]
[8, 9, 10, 11] [9, 10, 11, 12]
[9, 10, 11, 12] [10, 11, 12, 13]
[10, 11, 12, 13] [11, 12, 13, 14]
[11, 12, 13, 14] [12, 13, 14, 15]
[12, 13, 14, 15] [13, 14, 15, 16]
[13, 14, 15, 16] [14, 15, 16, 17]
[14, 15, 16, 17] [15, 16, 17, 18]
[15, 16, 17, 18] [16, 17, 18, 19]


In [131]:
def make_generator(toks, c_size):
    def gen():
        for i in range(len(toks)-c_size):
            yield toks[i:i+c_size], toks[i+1:i+c_size+1]    
    return gen

In [133]:
gen = make_generator(tokens, 4)
for x in gen():
    print(x)

([0, 1, 2, 3], [1, 2, 3, 4])
([1, 2, 3, 4], [2, 3, 4, 5])
([2, 3, 4, 5], [3, 4, 5, 6])
([3, 4, 5, 6], [4, 5, 6, 7])
([4, 5, 6, 7], [5, 6, 7, 8])
([5, 6, 7, 8], [6, 7, 8, 9])
([6, 7, 8, 9], [7, 8, 9, 10])
([7, 8, 9, 10], [8, 9, 10, 11])
([8, 9, 10, 11], [9, 10, 11, 12])
([9, 10, 11, 12], [10, 11, 12, 13])
([10, 11, 12, 13], [11, 12, 13, 14])
([11, 12, 13, 14], [12, 13, 14, 15])
([12, 13, 14, 15], [13, 14, 15, 16])
([13, 14, 15, 16], [14, 15, 16, 17])
([14, 15, 16, 17], [15, 16, 17, 18])
([15, 16, 17, 18], [16, 17, 18, 19])


In [ ]:
# ... do this instead
dataset = tf.data.Dataset.from_generator(
     gen,
     output_signature=(
         tf.TensorSpec(shape=(4), dtype=tf.int32),
         tf.TensorSpec(shape=(4), dtype=tf.int32)))
list(dataset.as_numpy_iterator())

## Try using tokenizer

In [142]:
text = "she had rings on her fingers and bells on her shoes"
tokens = tokenizer.encode(text)
print("tokens:", tokens)
# inputs = tf.data.Dataset.from_tensor_slices(tokens).window(4, shift=1, drop_remainder=True)
# targets = tf.data.Dataset.from_tensor_slices(tokens[1:]).window(4, shift=1, drop_remainder=True)
# dataset = tf.data.Dataset.zip(inputs, targets)
# for window in dataset:
#     print(list(window[0].as_numpy_iterator()), list(window[1].as_numpy_iterator()))
gen = make_generator(tokens, 4)
dataset = tf.data.Dataset.from_generator(
     gen,
     output_signature=(
         tf.TensorSpec(shape=(4), dtype=tf.int32),
         tf.TensorSpec(shape=(4), dtype=tf.int32)))
list(dataset.as_numpy_iterator())

tokens: [7091, 550, 13917, 319, 607, 9353, 290, 30987, 319, 607, 10012]


[(array([ 7091,   550, 13917,   319], dtype=int32),
  array([  550, 13917,   319,   607], dtype=int32)),
 (array([  550, 13917,   319,   607], dtype=int32),
  array([13917,   319,   607,  9353], dtype=int32)),
 (array([13917,   319,   607,  9353], dtype=int32),
  array([ 319,  607, 9353,  290], dtype=int32)),
 (array([ 319,  607, 9353,  290], dtype=int32),
  array([  607,  9353,   290, 30987], dtype=int32)),
 (array([  607,  9353,   290, 30987], dtype=int32),
  array([ 9353,   290, 30987,   319], dtype=int32)),
 (array([ 9353,   290, 30987,   319], dtype=int32),
  array([  290, 30987,   319,   607], dtype=int32)),
 (array([  290, 30987,   319,   607], dtype=int32),
  array([30987,   319,   607, 10012], dtype=int32))]

In [152]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
print("raw_text length:", len(raw_text))
tokens = tokenizer.encode(raw_text)
print("token count:", len(tokens))
print(raw_text[:1000])

raw_text length: 20479
token count: 5145
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)

"The height of his glory"--that was what the women called it. I can hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course it's going to send the value of my picture 'way up; but I don't think of that, Mr. Rickham--the loss to Arrt is all I think of." The word, on Mrs. Thwing's lips, multiplied its _rs_ as though they were reflected in an endless vista of mirrors. And it was not only the Mrs. Thwings who mourned. Had not the exquisite Hermia Croft, at the last Grafton Gallery show, stopped me before Gisburn's "Moon-dancers" to say, with tears in her eyes: "We shall not look upon

In [145]:
tokens = tokenizer.encode(raw_text[:100])
print("tokens:", tokens)

tokens: [40, 367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138, 257, 7026, 15632, 438, 2016, 257, 922, 5891, 1576, 438, 568, 340, 373, 645, 308]


In [ ]:
context_length=4
gen = make_generator(tokens, context_length)
dataset = tf.data.Dataset.from_generator(
     gen,
     output_signature=(
         tf.TensorSpec(shape=(context_length), dtype=tf.int32),
         tf.TensorSpec(shape=(context_length), dtype=tf.int32)))
list(dataset.as_numpy_iterator())

## Try our Dataset with a model

In [147]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=(context_length,)),
    tf.keras.layers.Embedding(tokenizer.vocab_size, 8),
    tf.keras.layers.Dense(units=10, activation='softmax', name="mydense")
    ], name="m1")
model.summary()    


Model: "m1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_5 (Embedding)     (None, 4, 8)              402056    
                                                                 
 mydense (Dense)             (None, 4, 10)             90        
                                                                 
Total params: 402146 (1.53 MB)
Trainable params: 402146 (1.53 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [148]:
model.compile(optimizer='sgd', loss=tf.keras.losses.KLDivergence(), metrics=['accuracy'])

In [ ]:
history = model.fit(x=dataset, epochs=10)